# BirdCLEF 2026 Training v40 — Perch + BirdNET Dual-Stream GRU

**Architecture**: concatenate Perch(1536) + BirdNET(1024) = 2560-d per window,
project to 512, then BiGRU + head (same as v23/v30).

**Fine-tuning strategy**:
1. Load v30 weights for GRU + head (skip the projection layer — input dim changed)
2. Freeze GRU + head for first 2 warmup epochs (train only the new projection)
3. Unfreeze all parameters for remaining epochs

**Kaggle inputs required:**
1. `birdclef-2026`
2. `chiragggg/birdclef-2026-perch-embs-v3`
3. `chiragggg/birdclef-2026-birdnet-embs-v1`
4. `chiragggg/birdclef-2026-perch-weights-v30`


In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, json, copy, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

CFG = dict(
    folds           = 5,
    epochs          = 15,
    freeze_epochs   = 2,        # freeze GRU+head, train only new projection
    warmup_epochs   = 1,
    lr              = 5e-5,     # higher lr for new projection layer
    lr_pretrained   = 5e-6,     # lower lr for GRU+head after unfreeze
    batch_size      = 4,
    num_workers     = 2,
    seed            = 42,
    perch_emb_dim   = 1536,
    birdnet_emb_dim = 1024,     # updated to actual dim after Cell 2
    perch_emb_noise = 0.005,
    birdnet_emb_noise = 0.005,
    gru_hidden      = 512,      # MUST match v23/v30
    gru_layers      = 2,        # MUST match v23/v30
    gru_dropout     = 0.3,
    max_seq_len     = 24,
    checkpoint_tag  = 'v40',
    device          = 'cuda' if torch.cuda.is_available() else 'cpu',
)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
device = torch.device(CFG['device'])

print('v40 Perch+BirdNET Dual-Stream GRU')
print(f'  Device : {device}')
print(f'  Epochs : {CFG["epochs"]}  (freeze={CFG["freeze_epochs"]})')
print(f'  LR     : proj={CFG["lr"]}  pretrained={CFG["lr_pretrained"]}')


In [ ]:
# === CELL 2: PATHS & AUTO-DETECT BIRDNET EMB DIM ===
def _fe(*c):
    return next((p for p in c if os.path.exists(p)), c[0])

TAXONOMY_CSV    = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                      '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
SOUNDSCAPE_ANNO = _fe('/kaggle/input/birdclef-2026/train_soundscapes_labels.csv',
                      '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')

PERCH_EMBD_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/birdclef-2026-perch-embs-v3',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
)
BIRDNET_EMBD_DIR = _fe(
    '/kaggle/input/birdclef-2026-birdnet-embs-v1/birdnet_embs',
    '/kaggle/input/birdclef-2026-birdnet-embs-v1',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-birdnet-embs-v1/birdnet_embs',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-birdnet-embs-v1',
)
V30_CKPT_DIR = _fe(
    '/kaggle/input/birdclef-2026-perch-weights-v30',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v30',
)
OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

# Auto-detect BirdNET emb dim from a sample file
_bn_files = list(Path(BIRDNET_EMBD_DIR).glob('birdnet_*.npy')) if os.path.isdir(BIRDNET_EMBD_DIR) else []
if _bn_files:
    _s = np.load(str(_bn_files[0]))
    CFG['birdnet_emb_dim'] = int(_s.shape[-1])
    print(f'BirdNET emb_dim auto-detected: {CFG["birdnet_emb_dim"]}')
else:
    print(f'WARNING: no birdnet_*.npy found in {BIRDNET_EMBD_DIR}')
    print(f'Using default emb_dim={CFG["birdnet_emb_dim"]}')

CFG['concat_dim'] = CFG['perch_emb_dim'] + CFG['birdnet_emb_dim']

_perch_n = len(list(Path(PERCH_EMBD_DIR).glob('soundscape_*.npy'))) if os.path.isdir(PERCH_EMBD_DIR) else 0
_v30_ckpts = list(Path(V30_CKPT_DIR).glob('perch_gru_v30_fold*.pt')) if os.path.isdir(V30_CKPT_DIR) else []
print(f'Species            : {n_classes}')
print(f'PERCH_EMBD_DIR     : {PERCH_EMBD_DIR}  ({_perch_n} files)')
print(f'BIRDNET_EMBD_DIR   : {BIRDNET_EMBD_DIR}  ({len(_bn_files)} files)')
print(f'concat_dim         : {CFG["concat_dim"]}')
print(f'V30_CKPT_DIR       : {V30_CKPT_DIR}  ({len(_v30_ckpts)} ckpts)')


In [ ]:
# === CELL 3: LABEL HELPERS ===
def soundscape_to_multihot(label_str):
    y = np.zeros(n_classes, dtype='float32')
    for sp in str(label_str).split(';'):
        sp = sp.strip()
        if sp in sp_idx: y[sp_idx[sp]] = 1.0
    return y

def _parse_hms(s):
    p = str(s).strip().split(':')
    return int(p[0])*3600 + int(p[1])*60 + int(p[2])

print('Label helpers defined')


In [ ]:
# === CELL 4: PERCHGRUDUAL MODEL ===
# Identical to PerchGRU (v23/v30) but takes concat(Perch, BirdNET) as input.
# The GRU + head are compatible with v30 weights.
class PerchGRUDual(nn.Module):
    def __init__(self, n_classes, concat_dim, hidden=512, n_layers=2, dropout=0.3):
        super().__init__()
        # New projection layer (NOT loaded from v30 — input dim changed)
        self.proj = nn.Sequential(
            nn.LayerNorm(concat_dim),
            nn.Linear(concat_dim, 512),
            nn.GELU(),
        )
        # These are loaded from v30 weights
        self.gru = nn.GRU(
            512, hidden, n_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )

    def forward(self, x):
        single = (x.dim() == 2)
        if single: x = x.unsqueeze(1)
        h, _ = self.gru(self.proj(x))
        out   = self.head(h)
        return out.squeeze(1) if single else out


_m = PerchGRUDual(n_classes, CFG['concat_dim']).to(device)
_x = torch.randn(2, 8, CFG['concat_dim']).to(device)
assert _m(_x).shape == (2, 8, n_classes)
del _m, _x
print(f'PerchGRUDual OK  concat_dim={CFG["concat_dim"]}')


In [ ]:
# === CELL 5: DUAL-STREAM DATASET ===
class DualStreamSeqDataset(Dataset):
    def __init__(self, seq_groups, perch_root, birdnet_root, train=True):
        self.groups       = seq_groups
        self.perch_root   = Path(perch_root)
        self.birdnet_root = Path(birdnet_root)
        self.train        = train

    def __len__(self): return len(self.groups)

    def __getitem__(self, i):
        grp     = self.groups[i]
        windows = sorted(grp['windows'], key=lambda w: w[1])[:CFG['max_seq_len']]
        T       = len(windows)
        feats   = np.zeros((T, CFG['concat_dim']), dtype='float32')
        labels  = np.zeros((T, n_classes), dtype='float32')
        for t, (sc_stem, end_secs, lv) in enumerate(windows):
            # Perch embedding
            pp = self.perch_root / f'soundscape_{sc_stem}_{end_secs}s.npy'
            if pp.exists():
                pe = np.load(str(pp)).astype('float32')
                if self.train and random.random() < 0.5:
                    pe += np.random.randn(*pe.shape).astype('float32') * CFG['perch_emb_noise']
            else:
                pe = np.zeros(CFG['perch_emb_dim'], dtype='float32')
            # BirdNET embedding
            bp = self.birdnet_root / f'birdnet_{sc_stem}_{end_secs}s.npy'
            if bp.exists():
                be = np.load(str(bp)).astype('float32')
                if self.train and random.random() < 0.5:
                    be += np.random.randn(*be.shape).astype('float32') * CFG['birdnet_emb_noise']
            else:
                be = np.zeros(CFG['birdnet_emb_dim'], dtype='float32')
            feats[t]  = np.concatenate([pe, be])
            labels[t] = lv
        return torch.from_numpy(feats), torch.from_numpy(labels)


def seq_collate(batch):
    xs, ys = zip(*batch)
    max_T  = max(x.shape[0] for x in xs)
    B      = len(xs)
    x_pad  = torch.zeros(B, max_T, CFG['concat_dim'])
    y_pad  = torch.zeros(B, max_T, n_classes)
    mask   = torch.zeros(B, max_T, dtype=torch.bool)
    for i,(x,y) in enumerate(zip(xs,ys)):
        T = x.shape[0]
        x_pad[i,:T] = x; y_pad[i,:T] = y; mask[i,:T] = True
    return x_pad, y_pad, mask

print('DualStreamSeqDataset + seq_collate defined')


In [ ]:
# === CELL 6: BUILD SEQUENCE GROUPS ===
sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
_sc_label_map = {}
for _, row in sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    _sc_label_map[(sc_stem, end_secs)] = soundscape_to_multihot(row['primary_label'])

_sc_groups = defaultdict(list)
for f in Path(PERCH_EMBD_DIR).glob('soundscape_*.npy'):
    try:
        stem_part, end_part = f.stem.rsplit('_', 1)
    except ValueError: continue
    if not end_part.endswith('s'): continue
    end_secs = int(end_part[:-1])
    sc_stem  = stem_part[len('soundscape_'):]
    lv       = _sc_label_map.get((sc_stem, end_secs))
    if lv is None: continue
    _sc_groups[sc_stem].append((sc_stem, end_secs, lv))

seq_groups = [{'stem':s,'windows':w} for s,w in _sc_groups.items() if w]
print(f'Sequences : {len(seq_groups)}')
print(f'Windows   : {sum(len(g["windows"]) for g in seq_groups)}')

# Check BirdNET coverage
_bn_found = sum(
    1 for g in seq_groups for (_,es,_) in g['windows']
    if (Path(BIRDNET_EMBD_DIR)/f'birdnet_{g["stem"]}_{es}s.npy').exists()
)
_total_w = sum(len(g['windows']) for g in seq_groups)
print(f'BirdNET coverage: {_bn_found}/{_total_w} windows ({100*_bn_found/_total_w:.1f}%)')
if _bn_found == 0:
    print('WARNING: No BirdNET embeddings found. Model will use zeros for BirdNET stream.')


In [ ]:
# === CELL 7: 5-FOLD FINE-TUNING ===
print('='*65)
print(f'v40 Dual-Stream  folds={CFG["folds"]}  AMP={torch.cuda.is_available()}')
print(f'freeze_epochs={CFG["freeze_epochs"]}  then full lr={CFG["lr_pretrained"]}')
print('='*65)

_use_amp   = (device.type == 'cuda')
_criterion = nn.BCEWithLogitsLoss(reduction='none')

sc_ds = DualStreamSeqDataset(seq_groups, PERCH_EMBD_DIR, BIRDNET_EMBD_DIR, train=True)
sc_dl = DataLoader(sc_ds, batch_size=CFG['batch_size'], shuffle=True,
                   num_workers=CFG['num_workers'], collate_fn=seq_collate,
                   drop_last=False, pin_memory=_use_amp)
print(f'DataLoader: {len(sc_ds)} soundscapes  {len(sc_dl)} batches/epoch')

fold_results = []

for fold_idx in range(CFG['folds']):
    v30_ckpt = Path(V30_CKPT_DIR) / f'perch_gru_v30_fold{fold_idx}.pt'
    if not v30_ckpt.exists():
        print(f'[SKIP] missing: {v30_ckpt}')
        continue

    print(f'\nFold {fold_idx+1}/{CFG["folds"]}  loading {v30_ckpt.name}')
    model = PerchGRUDual(
        n_classes, CFG['concat_dim'],
        CFG['gru_hidden'], CFG['gru_layers'], CFG['gru_dropout'],
    ).to(device)

    # Load v30 weights into GRU + head; skip proj (input dim changed)
    v30_state = torch.load(v30_ckpt, map_location=device, weights_only=True)
    own_state  = model.state_dict()
    loaded = 0; skipped = 0
    for k, v in v30_state.items():
        if k in own_state and own_state[k].shape == v.shape:
            own_state[k] = v; loaded += 1
        else:
            skipped += 1
    model.load_state_dict(own_state)
    print(f'  Loaded {loaded} tensors from v30  skipped {skipped} (proj)')

    # Phase 1: freeze GRU + head, train only proj
    for p in model.gru.parameters():  p.requires_grad = False
    for p in model.head.parameters(): p.requires_grad = False

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                      lr=CFG['lr'], weight_decay=1e-4)
    scaler    = GradScaler(enabled=_use_amp)

    best_loss  = float('inf')
    best_state = None

    for epoch in range(CFG['epochs']):
        # Unfreeze after freeze_epochs
        if epoch == CFG['freeze_epochs']:
            for p in model.gru.parameters():  p.requires_grad = True
            for p in model.head.parameters(): p.requires_grad = True
            # Rebuild optimizer with two param groups
            optimizer = AdamW([
                {'params': model.proj.parameters(), 'lr': CFG['lr']},
                {'params': list(model.gru.parameters()) + list(model.head.parameters()),
                 'lr': CFG['lr_pretrained']},
            ], weight_decay=1e-4)
            remaining = CFG['epochs'] - CFG['freeze_epochs'] - 1
            scheduler = CosineAnnealingLR(optimizer, T_max=max(1, remaining), eta_min=1e-7)
            print(f'  Epoch {epoch+1}: unfroze GRU+head, lr_pretrained={CFG["lr_pretrained"]}')

        model.train()
        ep_loss = 0.0; n_batches = 0

        for x_pad, y_pad, mask in tqdm(sc_dl, desc=f'  Ep {epoch+1}', leave=False):
            x_pad = x_pad.to(device)
            y_pad = y_pad.to(device)
            mask  = mask.to(device)

            optimizer.zero_grad()
            with autocast(enabled=_use_amp):
                logits = model(x_pad)
                loss_e = _criterion(logits, y_pad)
                m      = mask.unsqueeze(-1).float()
                loss   = (loss_e * m).sum() / m.sum().clamp(min=1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            ep_loss += loss.item(); n_batches += 1

        ep_loss /= max(n_batches, 1)
        if epoch >= CFG['freeze_epochs']:
            scheduler.step()

        if ep_loss < best_loss:
            best_loss  = ep_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f'  Ep {epoch+1:2d}/{CFG["epochs"]}  loss={ep_loss:.4f}')

    if best_state: model.load_state_dict(best_state)
    out_ckpt = os.path.join(OUT_DIR, f'perch_gru_v40_fold{fold_idx}.pt')
    torch.save(model.state_dict(), out_ckpt)
    fold_results.append(best_loss)
    print(f'  Saved {out_ckpt}  best_loss={best_loss:.4f}')

    del model, optimizer, scaler
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\nDone. Best losses: {["%.4f"%l for l in fold_results]}')
print(f'Saved: {sorted([f for f in os.listdir(OUT_DIR) if "v40" in f])}')


In [ ]:
# === CELL 8: UPLOAD AS birdclef-2026-perch-weights-v40 ===
import shutil, subprocess, json as _json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')
DATASET_SLUG    = 'birdclef-2026-perch-weights-v40'

_upload_dir = '/kaggle/working/upload_v40'
os.makedirs(_upload_dir, exist_ok=True)

_copied = []
for pt in Path(OUT_DIR).glob('perch_gru_v40_fold*.pt'):
    shutil.copy2(str(pt), os.path.join(_upload_dir, pt.name))
    _copied.append(pt.name)
print(f'Files to upload: {sorted(_copied)}')

if not _copied:
    print('ERROR: no v40 checkpoints found')
else:
    _meta = {'title':DATASET_SLUG,'id':f'{KAGGLE_USERNAME}/{DATASET_SLUG}',
             'licenses':[{'name':'CC0-1.0'}]}
    with open(os.path.join(_upload_dir,'dataset-metadata.json'),'w') as _mf:
        _json.dump(_meta,_mf,indent=2)
    _r = subprocess.run(['kaggle','datasets','create','-p',_upload_dir,'--dir-mode','zip'],
                        capture_output=True, text=True)
    print(_r.stdout)
    if _r.returncode != 0:
        print('STDERR:', _r.stderr)
        print(f'If exists: kaggle datasets version -p {_upload_dir} -m "v40 dual stream"')
    else:
        print(f'Upload complete: {KAGGLE_USERNAME}/{DATASET_SLUG}')
